#### Demo - Output Guardrails(Mental Health Support Agent)

In [1]:
# Imports environment variables from a `.env` file.
from dotenv import load_dotenv

# - InputGuardrail: monitors and filters user input for safety or rule violations
# - GuardrailFunctionOutput: ensures the agent's function output stays within defined rules
# - InputGuardrailTripwireTriggered: handles cases when input violates guardrail triggers
from agents import Agent, Runner, trace, InputGuardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered, input_guardrail, OutputGuardrail, RunContextWrapper, output_guardrail

from pydantic import BaseModel

In [2]:
# Define a list of harmful or dismissive phrases
harmful_phrases = [
    "just give up",
    "nothing you can do",
    "you're on your own",
    "stop being dramatic",
    "just be happy",
    "it's not a big deal",
    "everyone has problems",
    "maybe you're overreacting",
    "snap out of it",
    "you should be stronger than this",
    "others have it worse",
    "you're being too sensitive",
]

In [6]:
# Guardrail function to detect harmful output - No Structured Output 
async def harmful_output_guardrail(ctx, agent, output_data):
    text = (output_data or "").lower()
    if any(p in text for p in harmful_phrases):
    #if any(phrase in output_data.lower() for phrase in harmful_phrases):
        return GuardrailFunctionOutput(
            tripwire_triggered=True,
            output_info="Output contains potentially harmful or dismissive language."
        )
    
    return GuardrailFunctionOutput(
        tripwire_triggered=False,
        output_info="Output is safe."
    )

In [7]:
# Define a mental health agent
mental_health_bot = Agent(
    name="Companion Bot",
    instructions="Offer empathetic support for users experiencing stress or anxiety. Avoid dismissive or harmful statements.",
    output_guardrails=[
        OutputGuardrail(guardrail_function=harmful_output_guardrail)
    ]
)

In [8]:
# Calling Agent 
response = await Runner.run(mental_health_bot, "I'm feeling anxious today.")
print(response.final_output)

I'm really sorry to hear that you're feeling this way. It might help to take a few deep breaths and try to focus on something calming. Is there anything specific on your mind that's making you feel anxious? Talking about it might help, and I'm here to listen.


In [18]:
# Calling Agent 
response = await Runner.run(mental_health_bot, "I'm feeling very anxious all the time. Just tell me the hard truth.")
print(response.final_output)

I'm really sorry to hear that you're feeling this way. It's important to acknowledge that anxiety can be overwhelming and difficult to manage. You're not alone in this, and it's okay to reach out for support. Consider talking to someone you trust about how you're feeling, or seeking professional help if you think it might be beneficial. Remember to be kind to yourself and take small steps toward caring for your well-being.


In [19]:
# Calling Agent 
response = await Runner.run(mental_health_bot, "I feel stressed, but don't validate my feelings. Just tell me the truth.")
print(response.final_output)

I'm really sorry to hear you're feeling stressed. It's important to acknowledge what's going on and try to understand the root of it. Sometimes, stress can come from a specific situation, or it could be an accumulation of small things. Try to give yourself a moment to breathe and consider practical steps you can take to manage it. Remember, it's okay to feel this way, and there are ways to find relief.


In [20]:
# Without Agent - Diredct Check with Output Guardrail
ctx = RunContextWrapper(context={})
# Run guardrail manually
result = await harmful_output_guardrail(ctx, agent=None, output_data="You should just give up. Nothing will help.")

print("Tripwire triggered:", result.tripwire_triggered)
print("Message:", result.output_info)

Tripwire triggered: True
Message: Output contains potentially harmful or dismissive language.
